File for transforming the data and features that were deemed most accurate by logistic regression

In [2]:
# load all features

from datetime import datetime, date, timedelta
from google.colab import drive
import numpy as np
import pandas as pd

"""
To make these data import code lines work:

1. Go to "Honors Thesis" Shared Drive.
2. Click the three vertical dots to the right of the "relabeled_non_season_ending_injuries" folder.
3. Click "Organize."
4. Click "Add shortcut."
5. Click the "Add" button to the right of "My Drive."

These steps create a shortcut in My Drive which points to the folder of data.
"""

drive.mount('/content/drive')

# replace eda data with relabeled data from other folder

# all injuries resulting in less than 30 days of recovery are relabeled as non-injuries

eda1 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data1.csv")
eda2 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data2.csv")
eda3 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data3.csv")
eda4 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data4.csv")
eda5 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data5.csv")
eda6 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data6.csv")

# FILTER 1: IGNORE 2020-21 SEASON

# identify indices of the 2020-21 season and remove them from all six data tables

covid_season = ['2020-12-22', '2020-12-23', '2020-12-24', '2020-12-25',
                '2020-12-26', '2020-12-27', '2020-12-28', '2020-12-29',
                '2020-12-30', '2020-12-31', '2021-01-01', '2021-01-02',
                '2021-01-03', '2021-01-04', '2021-01-05', '2021-01-06',
                '2021-01-07', '2021-01-08', '2021-01-09', '2021-01-10',
                '2021-01-11', '2021-01-12', '2021-01-13', '2021-01-14',
                '2021-01-15', '2021-01-16', '2021-01-17', '2021-01-18',
                '2021-01-19', '2021-01-20', '2021-01-21', '2021-01-22',
                '2021-01-23', '2021-01-24', '2021-01-25', '2021-01-26',
                '2021-01-27', '2021-01-28', '2021-01-29', '2021-01-30',
                '2021-01-31', '2021-02-01', '2021-02-02', '2021-02-03',
                '2021-02-04', '2021-02-05', '2021-02-06', '2021-02-07',
                '2021-02-08', '2021-02-09', '2021-02-10', '2021-02-11',
                '2021-02-12', '2021-02-13', '2021-02-14', '2021-02-15',
                '2021-02-16', '2021-02-17', '2021-02-18', '2021-02-19',
                '2021-02-20', '2021-02-21', '2021-02-22', '2021-02-23',
                '2021-02-24', '2021-02-25', '2021-02-26', '2021-02-27',
                '2021-02-28', '2021-03-01', '2021-03-02', '2021-03-03',
                '2021-03-07', '2021-03-08', '2021-03-09', '2021-03-10',
                '2021-03-11', '2021-03-12', '2021-03-13', '2021-03-14',
                '2021-03-15', '2021-03-16', '2021-03-17', '2021-03-18',
                '2021-03-19', '2021-03-20', '2021-03-21', '2021-03-22',
                '2021-03-24', '2021-03-23', '2021-03-25', '2021-03-26',
                '2021-04-14', '2021-04-11', '2021-03-29', '2021-04-09',
                '2021-04-13', '2021-04-02', '2021-04-10', '2021-03-27',
                '2021-04-01', '2021-04-12', '2021-03-28', '2021-04-08',
                '2021-03-30', '2021-04-03', '2021-04-15', '2021-04-07',
                '2021-03-31', '2021-04-05', '2021-04-06', '2021-03-04',
                '2021-04-04', '2021-03-05', '2021-03-06', '2021-04-16']

removables = []

dates = list(eda1['date']) # rows of EDA data

for i in range(len(dates)): # for every row in EDA data...

  if dates[i] in covid_season:
    removables.append(i) # if the date aligns with the 2020-21 regular season, mark the row for filtering

  else:
    continue # if the date is not in the 2020-21 reguler season, check next row

eda1.drop(removables, inplace=True)
eda2.drop(removables, inplace=True)
eda3.drop(removables, inplace=True)
eda4.drop(removables, inplace=True)
eda5.drop(removables, inplace=True)
eda6.drop(removables, inplace=True)

balance = eda1['injured?'].value_counts() # distribution and positives and negatives in target variable

print(round(balance.values[0]/balance.values[1], 2), ' False values for every 1 True value')
balance

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-2116463695.py:26: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  eda1 = pd.read_csv("/content/drive/My Drive/relabeled_non_season_ending_injuries/season_ending_eda_data1.csv")


1197.86  False values for every 1 True value


,count
injured?,
False,491122
True,410


In [4]:
# derive extra features

def string_to_date(date_str): #convert strings to datetime
    return datetime.strptime(date_str, "%Y-%m-%d")

def date_to_string(date): #convert datetime to strings
    return date.strftime('%Y-%m-%d')

"""
rs_starts and rs_ends list the start and end of the regular seasons from 2017-18
to 2024-25. Each index from both lists is a single season.
"""
rs_starts = [
    datetime(2017, 10, 17),
    datetime(2018, 10, 16),
    datetime(2019, 10, 22),
    datetime(2020, 12, 22),
    datetime(2021, 10, 19),
    datetime(2022, 10, 18),
    datetime(2023, 10, 24),
    datetime(2024, 10, 22)
]

rs_ends = [
    datetime(2018, 4, 11),
    datetime(2019, 4, 10),
    datetime(2020, 4, 15),
    datetime(2021, 4, 16),
    datetime(2022, 4, 10),
    datetime(2023, 4, 9),
    datetime(2024, 4, 14),
    datetime(2025, 4, 13)
]

# add extra columns for time for testing:

eda1.head()

def get_month(date_str):
  month_num = date_str.split('-')[1]

  num_name = {
    "01": "Jan",
    "02": "Feb",
    "03": "Mar",
    "04": "Apr",
    "05": "May",
    "06": "Jun",
    "07": "Jul",
    "08": "Aug",
    "09": "Sep",
    "10": "Oct",
    "11": "Nov",
    "12": "Dec"
}

  return num_name[month_num]

eda1['month'] = eda1['date'].apply(get_month) # create new month column using the date

def get_season(date_str): # function for creating new season column
  date = string_to_date(date_str)

  seasons = [
    '2017-18',
    '2018-19',
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25'
  ]

  for i in range(len(seasons)):
    if (date >= rs_starts[i]) and (date <= rs_ends[i]):
      return seasons[i] # when the date appears within a season, return the season

eda1['season'] = eda1['date'].apply(get_season) # create new season column using the date

# new dataframe eda7 to examine how game_scheduled and games played affect injury

def false_to_yes(val):
  if val == 'False':
    return 1
  else:
    return 0

def find_game_day(val):
  if val == '-1':
    return 0
  else:
    return 1

game_days = eda2.iloc[:,56:112]
for i in range(56):
  game_days.iloc[:, i] = game_days.iloc[:, i].apply(false_to_yes)

# get list of strings of columns heads

cols = []
for i in range(56):
  cols.append('-' + str(i) + '_days_game_scheduled')

game_days.columns = cols

games_played = eda2.iloc[:,56:112]
for i in range(56):
  games_played.iloc[:, i] = games_played.iloc[:, i].apply(find_game_day)

# get list of strings of columns heads

cols = []
for i in range(56):
  cols.append('-' + str(i) + '_days_game_played')

games_played.columns = cols

eda7 = pd.concat([game_days, games_played, eda2['injured?']], axis=1)

In [33]:
# clean null values from data represented as -1.0

for col in list(eda1.columns)[164:192]:
  eda1[col] = eda1[col].apply(lambda x: 0 if x == -1.0 else x)

for col in list(eda2.columns)[1:56]:
  eda2[col] = eda2[col].apply(lambda x: 0 if x == -1.0 else x)

for col in list(eda3.columns)[57:101]:
  eda3[col] = eda3[col].apply(lambda x: 0 if x == -1.0 else x)

In [34]:
# features: month, game_scheduled, fga, minutes, injury status, arena latitude, arena longitude, target variable

data = eda1.loc[:, ['month']] # month

data = pd.concat([data, eda7.iloc[:,:9]], axis=1) # first 9 days of game_scheduled

sum_fga = eda3.iloc[:,57:101].T.sum()
sum_fga.name = 'sum_field_goals_attempted'
data = pd.concat([data, sum_fga], axis=1) # sum fga recent 44 days summed

sum_min = eda2.iloc[:,1:56].T.sum()
sum_min.name = 'sum_minutes'
data = pd.concat([data, sum_min], axis=1) # sum minutes recent 55 games

sum_injury_history = eda1.iloc[:,10:121].T.sum()
sum_injury_history.name = 'sum_injury_history'
data = pd.concat([data, sum_injury_history], axis=1) # sum injury history recent 55 games

data = pd.concat([data, eda1.iloc[:,164:192]], axis=1) # arena lat and arena long

data = pd.concat([data, eda1.loc[:, 'injured?']], axis=1) # target variable

print(data.shape)
data.head()

(491532, 42)


,month,-0_days_game_scheduled,-1_days_game_scheduled,-2_days_game_scheduled,-3_days_game_scheduled,-4_days_game_scheduled,-5_days_game_scheduled,-6_days_game_scheduled,-7_days_game_scheduled,-8_days_game_scheduled,...,-5_days_arena_long,-6_days_arena_long,-7_days_arena_long,-8_days_arena_long,-9_days_arena_long,-10_days_arena_long,-11_days_arena_long,-12_days_arena_long,-13_days_arena_long,injured?
0,Oct,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
1,Oct,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
2,Oct,1,0,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
3,Oct,1,1,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False
4,Oct,0,1,1,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False


In [36]:
data.to_csv('model_training_data.csv', index=False)